# bd4: 원천데이터 전체 영상 기반 fight / stalking / other 분류

이 노트북은 `C:\Users\우성한\Sample (1)\Sample\01.원천데이터` 아래의 모든 영상을 재귀적으로 탐색한 뒤, YOLO 기반 CF2 JSON을 생성하고 bd4 결과 JSON으로 분류합니다.

In [1]:
from pathlib import Path
import json
import math
import cv2
from tqdm import tqdm
from ultralytics import YOLO

# ===============================
# bd4 경로 설정
# ===============================
VIDEO_ROOT = Path(r"C:\Users\우성한\Sample (1)\Sample\01.원천데이터")
OUTPUT_ROOT = Path(r"C:\Users\우성한\outputs\prediction")

CF2_DIR = OUTPUT_ROOT / "cf2"
BD4_DIR = OUTPUT_ROOT / "bd4"

CF2_DIR.mkdir(parents=True, exist_ok=True)
BD4_DIR.mkdir(parents=True, exist_ok=True)

print("VIDEO_ROOT:", VIDEO_ROOT)
print("VIDEO_ROOT exists:", VIDEO_ROOT.exists())
print("CF2_DIR:", CF2_DIR)
print("BD4_DIR:", BD4_DIR)

if not VIDEO_ROOT.exists():
    raise RuntimeError(f"영상 루트 경로가 존재하지 않습니다: {VIDEO_ROOT}")

VIDEO_ROOT: C:\Users\우성한\Sample (1)\Sample\01.원천데이터
VIDEO_ROOT exists: True
CF2_DIR: C:\Users\우성한\outputs\prediction\cf2
BD4_DIR: C:\Users\우성한\outputs\prediction\bd4


In [2]:
# ===============================
# 원천데이터 전체 영상 탐색
# ===============================
VIDEO_EXTS = {
    ".mp4", ".avi", ".mov", ".mkv", ".wmv", ".mpeg", ".mpg", ".m4v",
    ".MP4", ".AVI", ".MOV", ".MKV", ".WMV", ".MPEG", ".MPG", ".M4V"
}

video_paths = [
    p for p in VIDEO_ROOT.rglob("*")
    if p.is_file() and p.suffix in VIDEO_EXTS
]

video_paths = sorted(video_paths)

print("찾은 영상 수:", len(video_paths))
for p in video_paths[:30]:
    print(p)

if len(video_paths) == 0:
    raise RuntimeError(
        "VIDEO_ROOT 아래에서 영상 파일을 찾지 못했습니다. "
        "확장자 또는 실제 폴더 구조를 확인하세요."
    )

찾은 영상 수: 22
C:\Users\우성한\Sample (1)\Sample\01.원천데이터\스토킹\특정 구역 내 지속 배회\lo_e1281\lo_e1281_c1.mp4
C:\Users\우성한\Sample (1)\Sample\01.원천데이터\스토킹\특정 구역 내 지속 배회\lo_e1281\lo_e1281_c2.mp4
C:\Users\우성한\Sample (1)\Sample\01.원천데이터\스토킹\특정 인물을 뒤따라가며 배회\fp_e1520\fp_e1520_c1.mp4
C:\Users\우성한\Sample (1)\Sample\01.원천데이터\스토킹\특정 인물을 뒤따라가며 배회\fp_e1520\fp_e1520_c2.mp4
C:\Users\우성한\Sample (1)\Sample\01.원천데이터\싸움\신체적 충돌을 동반한 싸움\ph_e2063\ph_e2063_c1.mp4
C:\Users\우성한\Sample (1)\Sample\01.원천데이터\싸움\신체적 충돌을 동반한 싸움\ph_e2063\ph_e2063_c2.mp4
C:\Users\우성한\Sample (1)\Sample\01.원천데이터\이륜 이동수단 인도 주행\오토바이 인도 주행\ms_e3500\ms_e3500_c1.mp4
C:\Users\우성한\Sample (1)\Sample\01.원천데이터\이륜 이동수단 인도 주행\오토바이 인도 주행\ms_e3500\ms_e3500_c2.mp4
C:\Users\우성한\Sample (1)\Sample\01.원천데이터\이륜 이동수단 인도 주행\자전거 인도 주행\bs_e4387\bs_e4387_c1.mp4
C:\Users\우성한\Sample (1)\Sample\01.원천데이터\이륜 이동수단 인도 주행\자전거 인도 주행\bs_e4387\bs_e4387_c2.mp4
C:\Users\우성한\Sample (1)\Sample\01.원천데이터\이륜 이동수단 인도 주행\전동킥보드 인도 주행\es_e3768\es_e3768_c1.mp4
C:\Users\우성한\Sample (1)\Sample\01.원천데

In [3]:
# ===============================
# YOLO 로드
# ===============================
model = YOLO("yolov8n.pt")
print("YOLO model loaded: yolov8n.pt")

YOLO model loaded: yolov8n.pt


In [4]:
# ===============================
# CF2 생성 함수
# ===============================
def safe_stem_from_path(path: Path, root: Path):
    """
    하위 폴더가 많아도 JSON 파일명이 충돌하지 않게 상대경로를 파일명으로 변환합니다.
    예: fight/001/a.mp4 -> fight__001__a.json
    """
    rel = path.relative_to(root)
    return "__".join(rel.with_suffix("").parts)


def create_cf2_from_video(video_path, output_json_path, frame_interval=15):
    cap = cv2.VideoCapture(str(video_path))

    if not cap.isOpened():
        raise RuntimeError(f"영상 열기 실패: {video_path}")

    frames = []
    frame_idx = 0
    sampled_idx = 0

    fps = cap.get(cv2.CAP_PROP_FPS)
    width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    total_frame_count = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))

    while True:
        ret, frame = cap.read()
        if not ret:
            break

        if frame_idx % frame_interval == 0:
            result = model(frame, verbose=False)[0]

            objects = []
            classes = {}

            if result.boxes is not None:
                for box in result.boxes:
                    cls_id = int(box.cls[0])
                    cls_name = model.names[cls_id]
                    conf = float(box.conf[0])
                    xyxy = box.xyxy[0].tolist()
                    bbox = [round(float(v), 2) for v in xyxy]

                    obj = {
                        "class": cls_name,
                        "confidence": round(conf, 4),
                        "bbox": bbox
                    }

                    objects.append(obj)
                    classes[cls_name] = classes.get(cls_name, 0) + 1

            frames.append({
                "frame_id": frame_idx,
                "sampled_id": sampled_idx,
                "object_count": len(objects),
                "classes": classes,
                "objects": objects
            })

            sampled_idx += 1

        frame_idx += 1

    cap.release()

    summary_classes = {}
    for fr in frames:
        for cls_name, count in fr["classes"].items():
            summary_classes[cls_name] = summary_classes.get(cls_name, 0) + count

    data = {
        "video_name": video_path.name,
        "video_path": str(video_path),
        "relative_video_path": str(video_path.relative_to(VIDEO_ROOT)),
        "video_info": {
            "fps": fps,
            "width": width,
            "height": height,
            "total_frame_count_from_capture": total_frame_count,
            "total_frames_read": frame_idx,
            "frame_interval": frame_interval,
            "sampled_frames": len(frames)
        },
        "summary": {
            "classes": summary_classes
        },
        "frames": frames
    }

    with open(output_json_path, "w", encoding="utf-8") as f:
        json.dump(data, f, ensure_ascii=False, indent=2)

    return data

In [5]:
# ===============================
# VIDEO_ROOT 전체 영상에 대해 CF2 생성/확인
# ===============================
created = 0
used_existing = 0
failed_cf2 = []

for video_path in tqdm(video_paths):
    stem = safe_stem_from_path(video_path, VIDEO_ROOT)
    output_json_path = CF2_DIR / f"{stem}.json"

    if output_json_path.exists():
        used_existing += 1
        continue

    try:
        create_cf2_from_video(
            video_path=video_path,
            output_json_path=output_json_path,
            frame_interval=15
        )
        created += 1

    except Exception as e:
        failed_cf2.append({
            "video": str(video_path),
            "error": str(e)
        })

print("\n================================")
print("CF2 생성/확인 완료")
print("전체 영상:", len(video_paths))
print("생성:", created)
print("기존 사용:", used_existing)
print("실패:", len(failed_cf2))
print("================================")

if failed_cf2:
    print("\n실패 목록 일부:")
    for item in failed_cf2[:20]:
        print(item)

100%|██████████████████████████████████████████████████████████████████████████████████| 22/22 [02:08<00:00,  5.86s/it]


CF2 생성/확인 완료
전체 영상: 22
생성: 22
기존 사용: 0
실패: 0


In [6]:
# ===============================
# bd4 분류 함수
# ===============================
def bbox_center(bbox):
    x1, y1, x2, y2 = bbox
    return ((x1 + x2) / 2, (y1 + y2) / 2)


def bbox_area(bbox):
    x1, y1, x2, y2 = bbox
    return max(0, x2 - x1) * max(0, y2 - y1)


def dist(p1, p2):
    return math.sqrt((p1[0] - p2[0]) ** 2 + (p1[1] - p2[1]) ** 2)


def extract_persons(frame):
    persons = []
    for obj in frame.get("objects", []):
        if obj.get("class") != "person":
            continue

        bbox = obj.get("bbox")
        if bbox is None or len(bbox) != 4:
            continue

        persons.append({
            "bbox": bbox,
            "center": bbox_center(bbox),
            "area": bbox_area(bbox),
            "confidence": obj.get("confidence", 0)
        })
    return persons


def classify_bd4_situation(data):
    frames = data.get("frames", [])

    if not frames:
        return {
            "label": "other",
            "reason": "분석 가능한 프레임 정보가 없음",
            "features": {},
            "scores": {
                "fight_score": 0,
                "stalking_score": 0,
                "other_score": 1
            }
        }

    person_frame_count = 0
    multi_person_frame_count = 0
    close_pair_count = 0
    very_close_pair_count = 0
    movement_events = 0
    large_movement_events = 0
    follow_like_events = 0
    persistent_person_events = 0

    prev_persons = []

    for frame in frames:
        persons = extract_persons(frame)

        if len(persons) > 0:
            person_frame_count += 1

        if len(persons) >= 2:
            multi_person_frame_count += 1
            centers = [p["center"] for p in persons]

            pair_distances = []
            for i in range(len(centers)):
                for j in range(i + 1, len(centers)):
                    pair_distances.append(dist(centers[i], centers[j]))

            if pair_distances:
                min_d = min(pair_distances)
                if min_d < 180:
                    close_pair_count += 1
                if min_d < 90:
                    very_close_pair_count += 1

        if prev_persons and persons:
            curr_centers = [p["center"] for p in persons]
            prev_centers = [p["center"] for p in prev_persons]

            # 프레임 간 인물 위치 변화량
            for pc in prev_centers:
                nearest_d = min(dist(pc, cc) for cc in curr_centers)
                if nearest_d > 50:
                    movement_events += 1
                if nearest_d > 100:
                    large_movement_events += 1

            # 사람이 계속 등장하는지
            if len(prev_persons) > 0 and len(persons) > 0:
                persistent_person_events += 1

            # 2명 이상이 비슷한 간격을 유지하며 이동하면 추적/동행 후보로 간주
            if len(prev_centers) >= 2 and len(curr_centers) >= 2:
                prev_sorted = sorted(prev_centers, key=lambda x: x[0])
                curr_sorted = sorted(curr_centers, key=lambda x: x[0])

                prev_gap = dist(prev_sorted[0], prev_sorted[1])
                curr_gap = dist(curr_sorted[0], curr_sorted[1])

                if abs(curr_gap - prev_gap) < 80:
                    follow_like_events += 1

        prev_persons = persons

    total_frames = len(frames)

    person_ratio = person_frame_count / total_frames
    multi_person_ratio = multi_person_frame_count / total_frames
    close_pair_ratio = close_pair_count / total_frames
    very_close_pair_ratio = very_close_pair_count / total_frames
    movement_ratio = movement_events / total_frames
    large_movement_ratio = large_movement_events / total_frames
    follow_like_ratio = follow_like_events / total_frames
    persistent_person_ratio = persistent_person_events / total_frames

    fight_score = 0
    stalking_score = 0

    # 싸움: 여러 사람 + 가까운 거리 + 큰 움직임 변화
    if multi_person_ratio >= 0.25:
        fight_score += 1
    if close_pair_ratio >= 0.15:
        fight_score += 1
    if very_close_pair_ratio >= 0.08:
        fight_score += 1
    if movement_ratio >= 0.25:
        fight_score += 1
    if large_movement_ratio >= 0.10:
        fight_score += 1

    # 스토킹: 사람이 지속 등장 + 이동 지속 + 추적/동행 형태 + 싸움식 초근접은 약함
    if person_ratio >= 0.35:
        stalking_score += 1
    if persistent_person_ratio >= 0.25:
        stalking_score += 1
    if movement_ratio >= 0.12:
        stalking_score += 1
    if follow_like_ratio >= 0.08:
        stalking_score += 1
    if very_close_pair_ratio < 0.10:
        stalking_score += 1

    # fight 우선 판정. 단, fight_score가 낮고 stalking_score가 높으면 stalking.
    if fight_score >= 3:
        label = "fight"
        reason = (
            "여러 사람이 함께 등장하는 비율이 높고, 인물 간 근접 또는 초근접 장면과 "
            "프레임 간 큰 움직임 변화가 관찰되어 싸움 후보로 판단"
        )
    elif stalking_score >= 3:
        label = "stalking"
        reason = (
            "사람이 지속적으로 등장하고 이동 흐름이 유지되며, 추적/동행 형태로 볼 수 있는 "
            "프레임 간 관계가 관찰되어 스토킹 후보로 판단"
        )
    else:
        label = "other"
        reason = (
            "싸움으로 볼 만큼의 다인 근접·급격한 움직임이 부족하고, "
            "스토킹으로 볼 만큼의 지속적 추적 움직임도 부족하여 기타 상황으로 판단"
        )

    return {
        "label": label,
        "reason": reason,
        "features": {
            "total_frames": total_frames,
            "person_frame_count": person_frame_count,
            "multi_person_frame_count": multi_person_frame_count,
            "close_pair_count": close_pair_count,
            "very_close_pair_count": very_close_pair_count,
            "movement_events": movement_events,
            "large_movement_events": large_movement_events,
            "follow_like_events": follow_like_events,
            "persistent_person_events": persistent_person_events,
            "person_ratio": round(person_ratio, 4),
            "multi_person_ratio": round(multi_person_ratio, 4),
            "close_pair_ratio": round(close_pair_ratio, 4),
            "very_close_pair_ratio": round(very_close_pair_ratio, 4),
            "movement_ratio": round(movement_ratio, 4),
            "large_movement_ratio": round(large_movement_ratio, 4),
            "follow_like_ratio": round(follow_like_ratio, 4),
            "persistent_person_ratio": round(persistent_person_ratio, 4)
        },
        "scores": {
            "fight_score": fight_score,
            "stalking_score": stalking_score
        }
    }

In [7]:
# ===============================
# CF2 전체 JSON을 bd4로 분류
# ===============================
cf2_json_paths = sorted(CF2_DIR.rglob("*.json"))

print("CF2 json 수:", len(cf2_json_paths))

if len(cf2_json_paths) == 0:
    raise RuntimeError("CF2 JSON이 0개입니다. CF2 생성 셀을 먼저 확인하세요.")

results = []
failed_bd4 = []

for json_path in tqdm(cf2_json_paths):
    try:
        with open(json_path, "r", encoding="utf-8") as f:
            data = json.load(f)

        pred = classify_bd4_situation(data)

        item = {
            "cf2_json": str(json_path),
            "video_name": data.get("video_name"),
            "video_path": data.get("video_path"),
            "relative_video_path": data.get("relative_video_path"),
            "label": pred["label"],
            "reason": pred["reason"],
            "features": pred.get("features", {}),
            "scores": pred.get("scores", {})
        }

        results.append(item)

        out_path = BD4_DIR / json_path.name
        with open(out_path, "w", encoding="utf-8") as f:
            json.dump(item, f, ensure_ascii=False, indent=2)

    except Exception as e:
        failed_bd4.append({
            "json": str(json_path),
            "error": str(e)
        })

summary = {
    "task": "bd4_fight_stalking_other_classification",
    "video_root": str(VIDEO_ROOT),
    "cf2_dir": str(CF2_DIR),
    "output_dir": str(BD4_DIR),
    "total_video": len(video_paths),
    "total_json": len(cf2_json_paths),
    "success_count": len(results),
    "failed_count": len(failed_bd4),
    "fight_count": sum(1 for r in results if r["label"] == "fight"),
    "stalking_count": sum(1 for r in results if r["label"] == "stalking"),
    "other_count": sum(1 for r in results if r["label"] == "other"),
    "results": results,
    "failed": failed_bd4,
    "cf2_failed": failed_cf2
}

summary_path = BD4_DIR / "bd4_batch_summary.json"

with open(summary_path, "w", encoding="utf-8") as f:
    json.dump(summary, f, ensure_ascii=False, indent=2)

print("\n================================")
print("bd4 전체 처리 완료")
print("================================")
print("전체 영상:", summary["total_video"])
print("전체 json:", summary["total_json"])
print("성공:", summary["success_count"])
print("실패:", summary["failed_count"])
print("fight:", summary["fight_count"])
print("stalking:", summary["stalking_count"])
print("other:", summary["other_count"])
print("summary 저장:", summary_path)

CF2 json 수: 22


100%|█████████████████████████████████████████████████████████████████████████████████| 22/22 [00:00<00:00, 324.92it/s]


bd4 전체 처리 완료
전체 영상: 22
전체 json: 22
성공: 22
실패: 0
fight: 7
stalking: 4
other: 11
summary 저장: C:\Users\우성한\outputs\prediction\bd4\bd4_batch_summary.json


In [8]:
# ===============================
# 어떤 파일이 어떤 상황인지 출력
# ===============================
summary_path = BD4_DIR / "bd4_batch_summary.json"

with open(summary_path, "r", encoding="utf-8") as f:
    summary = json.load(f)

results = summary["results"]

for label in ["fight", "stalking", "other"]:
    print("\n" + "=" * 70)
    print(label.upper())
    print("=" * 70)

    matched = [r for r in results if r["label"] == label]

    if not matched:
        print("해당 없음")
        continue

    for r in matched:
        print("video_name:", r.get("video_name"))
        print("relative_video_path:", r.get("relative_video_path"))
        print("video_path:", r.get("video_path"))
        print("label:", r.get("label"))
        print("reason:", r.get("reason"))
        print("scores:", r.get("scores"))
        print("features:", r.get("features"))
        print("-" * 70)


FIGHT
video_name: fp_e1520_c1.mp4
relative_video_path: 스토킹\특정 인물을 뒤따라가며 배회\fp_e1520\fp_e1520_c1.mp4
video_path: C:\Users\우성한\Sample (1)\Sample\01.원천데이터\스토킹\특정 인물을 뒤따라가며 배회\fp_e1520\fp_e1520_c1.mp4
label: fight
reason: 여러 사람이 함께 등장하는 비율이 높고, 인물 간 근접 또는 초근접 장면과 프레임 간 큰 움직임 변화가 관찰되어 싸움 후보로 판단
scores: {'fight_score': 4, 'stalking_score': 5}
features: {'total_frames': 62, 'person_frame_count': 33, 'multi_person_frame_count': 26, 'close_pair_count': 15, 'very_close_pair_count': 3, 'movement_events': 30, 'large_movement_events': 13, 'follow_like_events': 18, 'persistent_person_events': 31, 'person_ratio': 0.5323, 'multi_person_ratio': 0.4194, 'close_pair_ratio': 0.2419, 'very_close_pair_ratio': 0.0484, 'movement_ratio': 0.4839, 'large_movement_ratio': 0.2097, 'follow_like_ratio': 0.2903, 'persistent_person_ratio': 0.5}
----------------------------------------------------------------------
video_name: fp_e1520_c2.mp4
relative_video_path: 스토킹\특정 인물을 뒤따라가며 배회\fp_e1520\fp_e1520_c2.mp4
video_path

In [9]:
# ===============================
# 자연어 질의용 간단 검색 함수
# ===============================
def search_bd4_results(keyword=None, label=None):
    """
    예시:
    search_bd4_results(label="fight")
    search_bd4_results(keyword="stalking")
    search_bd4_results(keyword="폭행")
    """
    summary_path = BD4_DIR / "bd4_batch_summary.json"

    with open(summary_path, "r", encoding="utf-8") as f:
        summary = json.load(f)

    results = summary["results"]

    if label is not None:
        results = [r for r in results if r.get("label") == label]

    if keyword is not None:
        keyword_lower = keyword.lower()
        filtered = []
        for r in results:
            text = json.dumps(r, ensure_ascii=False).lower()
            if keyword_lower in text:
                filtered.append(r)
        results = filtered

    print("검색 결과 수:", len(results))

    for r in results:
        print("\nvideo_name:", r.get("video_name"))
        print("relative_video_path:", r.get("relative_video_path"))
        print("label:", r.get("label"))
        print("reason:", r.get("reason"))
        print("scores:", r.get("scores"))
        print("-" * 70)

    return results

# 사용 예시
# fight_results = search_bd4_results(label="fight")
# stalking_results = search_bd4_results(label="stalking")